# OpenAI API 실습 노트북

이 노트북은 **OpenAI API** 공개 강의자료를 Jupyter Notebook 형식으로 재구성한 자료입니다.

## 학습 목표

1. `uv`와 환경 변수로 OpenAI API 실습 환경을 구성한다.
2. Responses API로 텍스트 응답과 스트리밍 응답을 만든다.
3. 이미지 입력, 구조화된 출력, 웹 검색과 함수 도구 호출을 실습한다.
4. 이미지·음성 API와 토큰 사용량 관리 방법을 이해한다.

> **중요**: API 키는 절대 코드 셀이나 제출물에 직접 입력하지 말고 `.env` 파일에 보관하세요. 모델명·요금·지원 기능은 바뀔 수 있으므로 실제 수업 전 [공식 모델 문서](https://platform.openai.com/docs/models)와 [가격 문서](https://openai.com/api/pricing/)를 확인하세요.

## 1. OpenAI API란?

- 개발자가 GPT 모델을 애플리케이션, 웹사이트, 서비스에 통합하도록 돕는 인터페이스입니다.
- 사용자 입력을 보내면 모델이 응답을 생성하고, 대화형 기능을 구현할 수 있습니다.
- 공식 개발 문서: <https://platform.openai.com/docs/overview>

API 사용 전에는 결제 수단을 등록하고 API 크레딧을 준비해야 합니다. 사용량에 따라 비용이 차감되며, 크레딧이 부족하면 호출이 제한될 수 있습니다.

### 1-1. API 키 발급

1. <https://platform.openai.com/api-keys>에 접속합니다.
2. 새 비밀 키를 생성합니다.
3. 생성한 키는 한 번만 표시될 수 있으므로 안전한 곳에 보관합니다.
4. 키를 코드·노트북·GitHub에 직접 입력하지 않습니다.

### 1-2. 모델 선택 참고

아래 표는 원본 강의자료의 모델 선택 예시입니다. 실제 사용 가능 여부, 지원 기능과 요금은 계정 및 시점에 따라 달라질 수 있습니다.

| 선택 기준 | 예시 용도 |
| --- | --- |
| 저비용·대량 처리 | 요약, 분류, 간단한 챗봇 |
| 균형형 | RAG, 코딩, 도구 사용, 업무 자동화 |
| 고성능 | 복잡한 추론, 고난도 개발, 전문 에이전트 |

입력 토큰은 지시문·질문·대화 이력처럼 모델에 보내는 내용이고, 출력 토큰은 모델이 생성하는 답변입니다. 운영 서비스에서는 모델명을 환경 변수로 관리하는 편이 안전합니다.

## 2. `uv` 프로젝트 만들기

| 명령어 | 의미 |
| --- | --- |
| `uv init` | 프로젝트 생성 및 `pyproject.toml` 만들기 |
| `uv add 패키지명` | 프로젝트에 패키지 추가 |
| `uv remove 패키지명` | 패키지 제거 |
| `uv sync` | 의존성 설치 |
| `uv run 파일명.py` | 프로그램 실행 |
| `uv pip install` | 기존 `pip install`과 비슷한 방식 |

In [ ]:
# 터미널에서 한 번만 실행하세요.
# uv init
# uv add openai python-dotenv pydantic tiktoken
#
# Jupyter Notebook에서 실행 중이라면 현재 커널에 설치할 수도 있습니다.
# %pip install -U openai python-dotenv pydantic tiktoken

### 2-1. `.env` 파일 만들기

프로젝트 최상위 폴더에 `.env` 파일을 만든 뒤 다음처럼 작성합니다.

```text
OPENAI_API_KEY="나의 OpenAI API 키"
OPENAI_DEFAULT_MODEL="수업에서 사용할 모델명"
```

Git을 사용한다면 `.gitignore`에 다음 항목을 추가합니다.

```text
.env
.venv/
__pycache__/
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
default_model = os.getenv("OPENAI_DEFAULT_MODEL")

print("API 키 설정 여부:", "설정됨" if api_key else "설정되지 않음")
print("기본 모델:", default_model or "설정되지 않음")

if not api_key:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")
if not default_model:
    raise RuntimeError(".env 파일에 OPENAI_DEFAULT_MODEL을 설정한 뒤 다시 실행하세요.")

### 2-2. Jupyter 커널 선택

VS Code에서 `.ipynb` 파일을 열었다면 오른쪽 위의 **Select Kernel**을 선택하고, 프로젝트의 `.venv`에 해당하는 Python 환경을 선택합니다. 설치한 패키지가 있는 가상환경과 노트북 커널은 같아야 합니다.

## 3. OpenAI API 요청 만들기

Responses API는 현재 권장되는 통합 인터페이스입니다. 텍스트 생성, 이미지 입력, 도구 호출, 구조화된 출력 등을 같은 방식으로 처리할 수 있습니다.

In [2]:
from openai import OpenAI

client = OpenAI()  # OPENAI_API_KEY 환경 변수를 자동으로 읽습니다.

In [ ]:
response = client.responses.create(
    model=default_model,
    input="파이썬의 리스트를 비전공자도 이해할 수 있게 두 문장으로 설명해 줘.",
)

print(response.output_text)

인간의 언어와 컴퓨터의 언어는 다르다. EX) 오늘 날씨가 너무 좋아
1. 문장(오늘 날씨가 너무 좋아) -> 토큰화 ('오늘', '날씨', '가', '너무', '좋아)
2. '오늘','날씨','가' -> 숫자(컴퓨터의 언어)
    2-1. 딕셔너리, KEY(오늘), VALUE(숫자) -> (의미가 부여된 숫자로 변환)
    2-2. EMBDEDDING -> VALUE -> EMBEDDING VECTOR (의미가 부여된 숫자로 변환)
         DOG EAT HIM, HIM EAT DOG은 같은가? -> X 의미 전혀달라짐
3. EMBEDDING VECTOR + 위치 정보를 더해서 --> 최종 입력 데이터
4. TRANSFORMER DECODER BLOCK (문맥 이해할 수 있도록 해주는 것)
    4-1. MAKSED SELF-ATTENTION
         MASKED : 오늘 날씨 가 너무 좋아 / 좋아의 대상이 어디?
    4-2. 잔차연결 + 레이어정규화
         잔차열결 : 값이 사라질 수 있는데, 이를 방지
         레이어정규화 : 
    4-3. Feed Forward Network (맥락적 의미, 뉘앙스)
    4-4. 잔차연결 + 레이어정규화
         잔차연결 : 값이 사라질 수 있는데, 이를 방지
         레이어정규화 : 
5. Linear Layer : 오늘 날씨가 너무 좋아(logits, 숫자)
6. softmax : logtis -> 0과 1사의 값으로 변환(숫자)
7. 가장 큰 값을 선택해서 다음 문장으로 선택

### 3-1. Request Body 핵심 매개변수

| 매개변수 | 설명 | 예시 |
| --- | --- | --- |
| `model` | 사용할 AI 모델 | `model=default_model` |
| `input` | 질문, 대화 이력, 멀티모달 입력 | `input="AI란 무엇인가요?"` |
| `instructions` | 역할·말투·규칙 | `instructions="한국어 강사처럼 답하세요."` |
| `max_output_tokens` | 최대 출력 길이 | `max_output_tokens=300` |
| `stream` | 실시간 출력 여부 | `stream=True` |
| `tools` | 웹 검색 또는 함수 도구 | `tools=[...]` |
| `previous_response_id` | 이전 응답과 연결 | `previous_response_id=response.id` |
| `metadata` | 요청 분류·추적 정보 | `metadata={"user_id": "student_01"}` |
| `reasoning` | 지원 모델에서 추론 노력 설정 | `reasoning={"effort": "medium"}` |

In [ ]:
response = client.responses.create(
    model=default_model,
    instructions="당신은 초보자에게 쉽게 설명하는 파이썬 강사입니다.",
    input="RAG를 세 문장으로 설명해 주세요.",
    # 사용 모델이 지원하는 경우에만 활성화하세요.
    # reasoning={"effort": "medium"},
    max_output_tokens=300,
)

print(response.output_text)

### 3-2. 역할(role)과 `instructions`

- `user`: 사용자의 질문·요청
- `assistant`: 이전에 생성한 답변을 직접 전달할 때 사용
- `developer`: 서비스 운영 규칙·작업 지침
- `system`: 이전 메시지 방식에서 주로 쓰던 지시 역할

Responses API에서는 반복되는 공통 규칙을 최상위 `instructions`에 넣는 방식을 권장합니다.

In [ ]:
response = client.responses.create(
    model = default_model,
    input = [{'role':'user', 'content': '봄에 대해 알려줘'}]
)

print(response.output_text)

In [ ]:
response = client.responses.create(
    model = default_model,
    input = [{'role':'system', 
            'content':'너는 자바 언어를 이용하고 스프링 프레임워크를 이용한 웹 개발 전문가야'},
            {'role':'user',
            'content': '봄에 대해 알려줘'}]
)

print(response.output_text)

In [ ]:
response = client.responses.create(
    model = default_model,
    instructions="너는 자바 언어를 이용하고 스프링 프레임워크를 이용한 웹 개발 전문가야",
    input = '봄에 대해 알려줘',
    max_output_tokens = 1000
)

print(response.output_text)

## 4. 응답 받기: 일반 방식과 스트리밍

- 기본 방식(`stream=False`)은 응답 전체가 완성된 뒤 한 번에 받습니다.
- 스트리밍(`stream=True`)은 생성되는 텍스트 조각을 순서대로 받아 화면에 바로 보여 줍니다.
- 스트리밍은 첫 글자가 보일 때까지의 체감 시간을 줄일 수 있지만, 전체 생성 시간이 반드시 더 짧아지는 것은 아닙니다.

In [ ]:
response = client.responses.create(
    model=default_model,
    input="줄바꿈 없이 숫자를 콤마로 구분하여 1부터 100까지 세어 주세요.",
    max_output_tokens=150,
)

print(response.output_text)

In [ ]:
stream = client.responses.create(
    model=default_model,
    input="줄바꿈 없이 숫자를 콤마로 구분하여 1부터 100까지 세어 주세요.",
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

In [ ]:
# context manager 방식의 스트리밍 예시
with client.responses.stream(
    model=default_model,
    input="줄바꿈 없이 숫자를 콤마로 구분하여 1부터 100까지 세어 주세요.",
    max_output_tokens=150,
) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
print()

## 5. 이미지 이해하기(Vision)

공개 접근 가능한 이미지 URL 또는 업로드한 파일을 입력으로 전달할 수 있습니다. 인물 사진, 학생 과제, 개인정보가 있는 이미지는 동의와 기관 정책을 먼저 확인하세요.

In [ ]:
image_url = "https://images.unsplash.com/photo-1516321318423-f06f85e504b3?w=900"

response = client.responses.create(
    model=default_model,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "이 이미지의 핵심 내용을 한국어로 설명하고, 학습 자료로 쓸 때 주의할 점을 한 가지 알려 줘.",
                },
                {"type": "input_image", "image_url": image_url},
            ],
        }
    ],
)

print(response.output_text)

## 6. 구조화된 출력

구조화된 출력은 자유 문장 대신 프로그램이 바로 사용할 수 있는 일정한 데이터 구조를 받는 방법입니다. 분류, 정보 추출, 설문 분석, 챗봇 라우팅에 유용합니다.

In [ ]:
from pydantic import BaseModel, Field


class SentimentResult(BaseModel):
    label: str = Field(description="긍정, 부정, 중립 중 하나")
    confidence: int = Field(ge=0, le=100)
    reason: str


review = "배송은 빨랐지만 제품이 설명보다 작아서 아쉬웠어요."

response = client.responses.parse(
    model=default_model,
    input=[
        {"role": "developer", "content": "리뷰 감정을 분석하세요. confidence는 0~100 정수입니다."},
        {"role": "user", "content": review},
    ],
    text_format=SentimentResult,
)

result = response.output_parsed
print("분류:", result.label)
print("신뢰도:", result.confidence)
print("이유:", result.reason)

실제 서비스에서는 파싱 실패, 빈 값, 예외 처리도 반드시 고려해야 합니다.

## 7. 도구 사용

모델은 필요할 때 웹 검색 같은 내장 도구 또는 직접 만든 Python 함수를 호출할 수 있습니다.

In [ ]:
# 웹 검색 도구 지원 여부와 정확한 도구 이름은 사용하는 모델 및 SDK 문서를 확인하세요.
response = client.responses.create(
    model=default_model,
    tools=[{"type": "web_search"}],
    input="오늘 대한민국에서 발표된 AI 관련 주요 뉴스 2개를 출처와 함께 요약해 줘.",
)

print(response.output_text)

In [ ]:
import json


def get_weather(city: str) -> dict:
    """예제용 날씨 함수: 실제 서비스에서는 날씨 API를 연결합니다."""
    return {"city": city, "weather": "맑음", "temperature": 22}


tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "도시의 현재 날씨를 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
            "additionalProperties": False,
        },
    }
]

first = client.responses.create(
    model=default_model,
    input="서울 날씨를 알려 줘.",
    tools=tools,
)

outputs = []
for item in first.output:
    if item.type == "function_call" and item.name == "get_weather":
        result = get_weather(**json.loads(item.arguments))
        outputs.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            }
        )

final = client.responses.create(
    model=default_model,
    previous_response_id=first.id,
    input=outputs,
)

print(final.output_text)

## 8. 멀티모달 입력: 텍스트 + 이미지

`content` 배열에 `input_text`, `input_image`, `input_audio`, `input_file` 같은 입력을 함께 넣을 수 있습니다. 아래 예시는 로컬 이미지 파일을 Base64 문자열로 인코딩해 전달합니다.

In [ ]:
import base64


def encode_image(path: str) -> str:
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# 같은 폴더에 sample.png를 준비한 뒤 실행하세요.
image_base64 = encode_image("sample.png")

response = client.responses.create(
    model=default_model,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "이 이미지가 무엇인지 설명해 주세요."},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{image_base64}",
                },
            ],
        }
    ],
    max_output_tokens=300,
)

print(response.output_text)

### 입력 타입 요약

| type | 용도 | 주요 값 |
| --- | --- | --- |
| `input_text` | 질문·지시 | `text` |
| `input_image` | 이미지 분석, OCR, 차트 해석 | `image_url`, `detail` |
| `input_audio` | 음성 질문·분석 | Base64 `data`, `format` |
| `input_file` | 업로드 파일 참조 | `file_id` |

## 9. 다양한 모델 사용 예시

아래 예시는 이미지 생성·편집, TTS, STT의 기본 흐름을 보여 줍니다. 모델의 지원 여부 및 세부 옵션은 실행 전 공식 문서를 확인하세요.

In [ ]:
# 이미지 생성
from openai import OpenAI
import base64

client = OpenAI()

response = client.images.generate(
    model="gpt-image-2",
    prompt="겨울 우주를 배경으로 한 밝고 귀여운 마녀 캐릭터 일러스트를 만들어 줘.",
    size="1024x1024",
)

image_base64 = response.data[0].b64_json
with open("step1.png", "wb") as file:
    file.write(base64.b64decode(image_base64))

print("step1.png 저장 완료")

In [ ]:
# 이미지 편집: 앞 셀에서 생성한 step1.png가 있어야 합니다.
with open("step1.png", "rb") as image_file:
    response = client.images.edit(
        model="gpt-image-2",
        image=image_file,
        prompt="배경을 신비로운 보랏빛 숲으로 바꿔줘.",
    )

image_base64 = response.data[0].b64_json
with open("step2.png", "wb") as file:
    file.write(base64.b64decode(image_base64))

print("step2.png 저장 완료")

In [ ]:
from openai import OpenAI
import base64

client = OpenAI()

with open("step2.png", "rb") as image_file:

    response = client.images.edit(
        model="gpt-image-2",
        image=image_file,
        prompt="""
다음 제목 텍스트를 추가하세요.

"행복한 할로윈!!"

이미지 상단 중앙에 배치하세요.

텍스트는 크고 우아하며 읽기 쉽도록 만들어 주세요.
한글을 사용해 주세요.
        """
    )

image_base64 = response.data[0].b64_json

with open("step3.png", "wb") as f:
    f.write(base64.b64decode(image_base64))

print("step3.png 저장 완료")

In [ ]:
# 텍스트를 음성으로 변환(TTS)
text = """
안녕하세요. 저는 OpenAI 음성 모델입니다.
텍스트를 자연스러운 한국어 음성으로 변환하고 있습니다.
"""

response = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="nova",
    input=text,
)

with open("korean_voice.mp3", "wb") as file:
    file.write(response.read())

print("korean_voice.mp3 저장 완료")

In [ ]:
# 음성을 텍스트로 변환(STT): 앞 셀에서 만든 korean_voice.mp3를 사용합니다.
with open("korean_voice.mp3", "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=audio_file,
    )

with open("transcript.txt", "w", encoding="utf-8") as file:
    file.write(transcript.text)

print(transcript.text)

## 10. 토큰과 비용 관리

토큰은 모델이 텍스트를 처리하는 기본 단위입니다. 영어는 대략 4글자 정도가 1토큰인 경우가 많고, 한국어는 문장에 따라 2~3글자 정도가 1토큰일 수 있습니다.

- 토크나이저: <https://platform.openai.com/tokenizer>
- 비용·사용량은 Platform 대시보드에서 확인합니다.

In [ ]:
import tiktoken


def get_token_count(text: str, model: str = "gpt-4o-mini") -> int:
    """지정한 모델의 인코딩으로 대략적인 토큰 수를 계산합니다."""
    return len(tiktoken.encoding_for_model(model).encode(text))


text = "줄바꿈 없이 숫자를 콤마로 구분하고 1부터 100까지 숫자를 세어줘."
print(get_token_count(text))

In [ ]:
# 실제 응답의 사용량은 Response 객체의 usage에서 확인할 수 있습니다.
response = client.responses.create(
    model=default_model,
    input="안녕하세요",
)

print("사용량 정보:", response.usage)

## 11. 미니 과제 1 — 초간단 챗봇

### 과제 목표

1. **Single Turn**: 한 번 질문하고 한 번 답하는 챗봇을 만듭니다.
2. **Multi Turn**: 이전 대화 맥락을 유지하지 못합니다.
3. **Multi Turn**: `previous_response_id`를 사용해 이전 대화 맥락을 유지합니다.
4. `quit`를 입력하면 프로그램이 종료되게 만듭니다.

추천 질문:

- “서울에 있는 유명한 전통음식점 3곳만 알려줘.”
- “네가 알려준 식당 중 한 곳은 서울역에서 대중교통으로 어떻게 가?”

In [ ]:
# Single Turn 챗봇


In [ ]:
# multi-turn 챗봇 예제(맥락 이어지지 않음)


In [ ]:
# Multi Turn 챗봇: 이전 응답 ID를 이어서 전달합니다.


| 구분 | Single Turn | Multi Turn |
| --- | --- | --- |
| 대화 기억 | 없음 | 있음 |
| 맥락 이해 | 불가능 | 가능 |
| 구현 난이도 | 쉬움 | 약간 복잡 |
| 사용 사례 | FAQ, 단순 질의 | 챗봇, 상담, AI 비서 |

## 12. 미니 과제 2

### 12-1. 친절한 과목 도우미

- 역할: 파이썬 초보 학습 도우미
- 답변: 한국어, 3문장 이내
- 출력: `response.output_text`

### 12-2. 영화 리뷰 감정 분석

Pydantic 구조화 출력을 사용해 `긍정 / 부정 / 중립`, 신뢰도(0~100), 이유를 반환하세요.

입력 예시: `배우의 연기는 좋았지만 전개가 너무 느려서 지루했습니다.`

### 12-3. 이미지 설명 도우미

이미지 URL을 입력받아 다음 형식으로 출력하세요.

1. 핵심 대상
2. 분위기
3. 접근성용 대체 텍스트

## 참고 자료

- [OpenAI API 문서](https://platform.openai.com/docs/overview)
- [모델 목록과 기능](https://platform.openai.com/docs/models)
- [Responses API](https://platform.openai.com/docs/api-reference/responses)
- [가격](https://openai.com/api/pricing/)
- [API 사용량 Dashboard](https://platform.openai.com/usage)